In [ ]:
import gc
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl

# Configure Polars print outs
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)

In [ ]:
INPUT_DATA_DIR = Path("./data")
RAW_INPUT_DATA_DIR = INPUT_DATA_DIR / "raw"
FEATURES_DATA_PATH = INPUT_DATA_DIR / "m5-acc-features-df.pq"

In [ ]:
# Data constants
LABEL_COL_NAME = "sales"
MAX_TRAIN_INDEX = 1941
FORECAST_HORIZON = 28
VALIDATION_DAYS = 28
VALID_START_INDEX = MAX_TRAIN_INDEX - VALIDATION_DAYS + 1

# Only the most recent MAX_TRAIN_DAYS days are used for training + validation.
# Lower this to reduce RAM usage (None = use everything).
MAX_TRAIN_DAYS: int | None = 1000

# Feature engineering constants. These must match what the feature engineering
# notebook (1.1) used to build the features parquet.
SALES_LAGS = [1, 2, 3, 4, 7, 14]
ROLLING_SALES_WINDOWS = [2, 3, 4, 7, 14]
PRICE_LAGS = [1, 2, 3, 4, 7]
ROLLING_PRICE_WINDOWS = [2, 3, 4, 7, 14]
N_EVENTS = 2
SNAP_STATES = ["CA", "TX", "WI"]
MAX_DAYS_UNTIL_EVENT = 7
MAX_DAYS_SINCE_EVENT = 7
MAX_DAYS_UNTIL_ELECTION = 7
MAX_DAYS_SINCE_ELECTION = 7
CYCLICAL_CALENDAR_FEATURE_SUFFIXES = ["wday", "month"]
ELECTION_COLS = [f"snap_{s}" for s in SNAP_STATES]

### Feature configuration

In [ ]:
# Feature configuration
ALL_FEATURE_GROUPS: dict[str, list[str]] = {
    "id_categorical": ["item_id", "dept_id", "cat_id", "store_id", "state_id"],

    # Sales features
    "sales_lags": [f"sales_lagged_{lag}" for lag in SALES_LAGS],
    "sales_rolling_mean": [f"sales_rolling_mean_{w}" for w in ROLLING_SALES_WINDOWS],
    
    # Price features
    "sell_price": ["sell_price"],
    "price_lags": [f"sell_price_lagged_{lag}" for lag in PRICE_LAGS],
    "price_rolling_mean": [f"sell_price_rolling_mean_{w}" for w in ROLLING_PRICE_WINDOWS],
    "price_rolling_std": [f"sell_price_rolling_std_{w}" for w in ROLLING_PRICE_WINDOWS],
    
    # Calendar features
    "calendar_cyclical": [
        f"{trig}_{suffix}"
        for suffix in CYCLICAL_CALENDAR_FEATURE_SUFFIXES
        for trig in ("sin", "cos")
    ],
    "calendar_raw": ["wday", "month", "year"],
    "is_weekend": ["is_weekend"],

    # Event features
    "event_categorical": [
        f"{prefix}_{i}"
        for prefix in ("event_name", "event_type", "last_event_type", "next_event_type")
        for i in range(1, N_EVENTS + 1)
    ],
    "event_timing": [
        f"days_{direction}_event_{i}"
        for direction in ("since", "until")
        for i in range(1, N_EVENTS + 1)
    ],

    # Election features
    "snap": list(ELECTION_COLS),
    "election_timing": [
        f"days_{direction}_{e}" 
        for direction in ("since", "until") 
        for e in ELECTION_COLS
    ],
}
CATEGORICAL_FEATURE_GROUPS = {"id_categorical", "event_categorical"}

# ---- Configure features ----
# Add / remove groups to include / drop all of its features from training and inference.
FEATURE_GROUPS_TO_INCLUDE = [
    "id_categorical",
    "event_categorical",
    "sell_price",
    "sales_lags",
    "sales_rolling_mean",
    "price_lags",
    "price_rolling_mean",
    "price_rolling_std",
    "calendar_cyclical",
    # "calendar_raw",
    "is_weekend",
    "event_timing",
    "snap",
    "election_timing",
]

# Add / remove individual features
EXTRA_FEATURES_TO_INCLUDE: list[str] = []
FEATURES_TO_EXCLUDE: list[str] = []
# ------------------------

# Construct features
_categorical_candidates = {c for g in CATEGORICAL_FEATURE_GROUPS for c in ALL_FEATURE_GROUPS[g]}
FEATURE_COLS = list(
    dict.fromkeys(
        c
        for c in [*(c for g in FEATURE_GROUPS_TO_INCLUDE for c in ALL_FEATURE_GROUPS[g]), *EXTRA_FEATURES_TO_INCLUDE]
        if c not in FEATURES_TO_EXCLUDE
    )
)
CATEGORICAL_FEATURE_COLS = [c for c in FEATURE_COLS if c in _categorical_candidates]

print(f"{len(FEATURE_COLS)} features ({len(CATEGORICAL_FEATURE_COLS)} categorical)")

### Model configuration

In [ ]:
# LightGBM config
NUM_BOOST_ROUND = 10_000
EARLY_STOPPING_ROUNDS = 10

LGB_PARAMS = {
    "objective": "poisson",
    "metric": "rmse",
    "learning_rate": 0.02,
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "cat_smooth": 20,
    "force_row_wise": True,
    "max_bin": 255,
    "seed": 0,
    "num_threads": -1,
    "verbosity": -1,
}

### Load data

In [ ]:
DATA_COLS = list(dict.fromkeys(["id", "d_index", LABEL_COL_NAME, *FEATURE_COLS]))
MIN_TRAIN_INDEX = 0 if MAX_TRAIN_DAYS is None else MAX_TRAIN_INDEX - MAX_TRAIN_DAYS

full_df = (
    pl.scan_parquet(FEATURES_DATA_PATH)
    .select(DATA_COLS)
    .filter(pl.col("d_index") > MIN_TRAIN_INDEX)
    .with_columns(pl.col(pl.Float64).cast(pl.Float32))
    .collect()
)
print(f"{full_df.estimated_size('gb'):.2f} GB")
full_df.shape

In [ ]:
# For submission, features are constructed iteratively. Keep only the final slice of
# the sales / price history needed to construct features for the first forecast day,
# so the full feature df isn't needed at inference.
max_lookback = max(SALES_LAGS + ROLLING_SALES_WINDOWS + PRICE_LAGS + ROLLING_PRICE_WINDOWS)

history_df = (
    pl.scan_parquet(FEATURES_DATA_PATH)
    .select(["id", "d_index", "sales", "sell_price"])
    .filter(pl.col("d_index") > MAX_TRAIN_INDEX - max_lookback)
    .with_columns(pl.col("sell_price").cast(pl.Float32))
    .sort(["id", "d_index"])
    .collect()
)
sales_history_df = history_df.select(["id", "d_index", "sales"])
price_history_df = history_df.select(["id", "d_index", "sell_price"])
del history_df
gc.collect()

sales_history_df.head(max_lookback + 2)

## Prepare data

In [ ]:
# Construct category mapping from training data only.
CATEGORY_DTYPES = {
    col: pl.Enum(
        full_df.filter(pl.col("d_index") < VALID_START_INDEX)[col]
        .cast(pl.String).drop_nulls().unique().sort()
    )
    for col in CATEGORICAL_FEATURE_COLS
}


def to_float32_matrix(df: pl.DataFrame) -> np.ndarray:
    """
    Feature matrix (float32, NaN = missing) in FEATURE_COLS order.
    Categoricals become integer codes (LightGBM treats them as categories through `categorical_feature`).
    """
    X = np.empty((df.height, len(FEATURE_COLS)), dtype=np.float32)
    for j, col in enumerate(FEATURE_COLS):
        s = df.get_column(col)
        if col in CATEGORY_DTYPES:
            s = s.cast(pl.String).cast(CATEGORY_DTYPES[col], strict=False).to_physical()
        X[:, j] = s.cast(pl.Float32).to_numpy()
    return X


In [ ]:
is_valid = pl.col("d_index") >= VALID_START_INDEX
train_df = full_df.filter(~is_valid)
valid_df = full_df.filter(is_valid)

X_train, y_train = to_float32_matrix(train_df), train_df[LABEL_COL_NAME].to_numpy().astype(np.float32)
X_valid, y_valid = to_float32_matrix(valid_df), valid_df[LABEL_COL_NAME].to_numpy().astype(np.float32)
print(X_train.shape, X_valid.shape)

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    feature_name=FEATURE_COLS,
    categorical_feature=CATEGORICAL_FEATURE_COLS,
    params=LGB_PARAMS,
)
valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    reference=train_set,
    feature_name=FEATURE_COLS,
    categorical_feature=CATEGORICAL_FEATURE_COLS,
    params=LGB_PARAMS,
)

## Model

In [ ]:
evals_result: dict = {}
model = lgb.train(
    LGB_PARAMS,
    train_set,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[valid_set],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(evals_result),
    ],
)
print(f"Best iteration: {model.best_iteration}, "
      f"valid RMSE: {model.best_score['valid']['rmse']:.4f}")

In [ ]:
(
    pl.DataFrame({
        "feature": model.feature_name(),
        "gain": model.feature_importance(importance_type="gain"),
    })
    .sort("gain", descending=True)
    .with_columns(gain_share=pl.col("gain") / pl.col("gain").sum())
)

## Submission

In [ ]:
# Load data needed for inference
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in SNAP_STATES},
}
raw_calendar_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

raw_sell_price_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "sell_prices.csv",
    schema_overrides={"wm_yr_wk": pl.Int16, "sell_price": pl.Float32},
)
sample_submission_df = pl.read_csv(RAW_INPUT_DATA_DIR / "sample_submission.csv")

In [ ]:
# Calendar / event / election features are recomputed for inference. The logic is
# the same as in the feature engineering notebook (1.1).
event_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(f"event_name_{i}").is_not_null())
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"event_index_{i}")
        for i in range(1, N_EVENTS + 1)
    )
    .with_columns(
        *[
            (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
            .alias(f"days_since_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.col(f"event_type_{i}").forward_fill().alias(f"last_event_type_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.col(f"event_type_{i}").backward_fill().alias(f"next_event_type_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .with_columns(
        # Clip days since / until events (and the associated event types)
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT)
            .then(pl.col(f"days_since_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT)
            .then(pl.col(f"last_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"days_until_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"next_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
    )
)

calendar_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float32),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float32),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float32),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float32),
        is_weekend=(pl.col("wday") <= 2).cast(pl.Float32),
    )
)

election_features_df = (
    raw_calendar_df
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(election).eq(1))
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"{election}_index")
        for election in ELECTION_COLS
    )
    .select(["wm_yr_wk", "date", "d", "d_index"] + ELECTION_COLS + [f"{e}_index" for e in ELECTION_COLS])
    .with_columns(
        *[
            (pl.col("d_index") - pl.col(f"{e}_index").forward_fill()).alias(f"days_since_{e}")
            for e in ELECTION_COLS
        ],
        *[
            (pl.col(f"{e}_index").backward_fill() - pl.col("d_index")).alias(f"days_until_{e}")
            for e in ELECTION_COLS
        ],
    )
    .with_columns(
        *[
            pl.when(pl.col(f"days_since_{e}") <= MAX_DAYS_SINCE_ELECTION)
            .then(pl.col(f"days_since_{e}"))
            .otherwise(None)
            for e in ELECTION_COLS
        ],
        *[
            pl.when(pl.col(f"days_until_{e}") <= MAX_DAYS_UNTIL_ELECTION)
            .then(pl.col(f"days_until_{e}"))
            .otherwise(None)
            for e in ELECTION_COLS
        ],
    )
    .drop([f"{e}_index" for e in ELECTION_COLS])
)

# Merge once, and keep only the forecast days (small).
forecast_calendar_df = (
    calendar_features_df.drop("d", "date")
    .join(event_features_df.drop("d", "date"), on=["wm_yr_wk", "d_index"], how="inner", validate="1:1")
    .join(election_features_df.drop("d", "date"), on=["wm_yr_wk", "d_index"], how="inner", validate="1:1")
    .filter(pl.col("d_index") > MAX_TRAIN_INDEX)
)
forecast_calendar_df.head()

In [ ]:
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

_SERIES_ID_RE = (
    r"^(?P<cat>[A-Z]+)_(?P<dept>\d+)_(?P<item>\d+)_(?P<state>[A-Z]+)_(?P<store>\d+)"
    r"_(?:validation|evaluation)$"
)

FORECAST_IDS = (
    sample_submission_df
    .filter(sample_submission_df["id"].str.ends_with("_evaluation"))
    ["id"]
    .unique()
    .to_list()
)

In [ ]:
# Recursive forecasting: iterate over forecast days, predict, and feed the predictions
# back in as sales history for the features of the next day.
all_forecast_dfs: list[pl.DataFrame] = []

for f_index in range(1, FORECAST_HORIZON + 1):
    d_index = MAX_TRAIN_INDEX + f_index

    # Construct test_df with identifier columns
    test_df = (
        pl.DataFrame({"id": FORECAST_IDS})
        .with_columns(parsed_id=pl.col("id").str.extract_groups(_SERIES_ID_RE))
        .with_columns(
            item_id=pl.format(
                "{}_{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
                pl.col("parsed_id").struct.field("item"),
            ),
            dept_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
            ),
            cat_id=pl.col("parsed_id").struct.field("cat"),
            store_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("state"),
                pl.col("parsed_id").struct.field("store"),
            ),
            state_id=pl.col("parsed_id").struct.field("state"),
        )
        .drop("parsed_id")
        .with_columns(
            id=pl.col("id").cast(pl.Categorical),
            d_index=pl.lit(d_index).cast(pl.Int16),
            sales=pl.lit(None).cast(pl.Int16),
        )
        # Prices for the current d_index are known in advance
        .join(
            other=forecast_calendar_df.select(["d_index", "wm_yr_wk"]),
            on="d_index", how="left", validate="m:1",
        )
        .join(
            other=raw_sell_price_df,
            on=["item_id", "store_id", "wm_yr_wk"], how="left", validate="m:1",
        )
        .drop("wm_yr_wk")
    )

    # Lagged / rolling sales features
    previous_sales_df = (
        pl.concat(
            [
                sales_history_df,
                *[
                    df.select(["id", "d_index", "sales"])
                    .with_columns(id=pl.col("id").cast(pl.Categorical))
                    for df in all_forecast_dfs
                ],
                test_df.select(["id", "d_index", "sales"]),
            ]
        )
        .sort(by=["id", "d_index"])
    )

    test_sales_features_df = (
        previous_sales_df
        .with_columns(
            [
                pl.col("sales").shift(lag).over("id", order_by="d_index")
                .alias(f"sales_lagged_{lag}")
                for lag in SALES_LAGS
            ]
            + [
                pl.col("sales").shift(1)
                .rolling_mean(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sales_rolling_mean_{window}")
                .cast(pl.Float32)
                for window in ROLLING_SALES_WINDOWS
            ]
        )
        .drop("sales")
        .filter(pl.col("d_index") == d_index)
    )

    # Lagged / rolling price features
    price_history_df = (
        pl.concat([price_history_df, test_df.select(["id", "d_index", "sell_price"])])
        .sort(by=["id", "d_index"])
    )

    test_price_features_df = (
        price_history_df
        .with_columns(
            [
                pl.col("sell_price").shift(lag).over("id", order_by="d_index")
                .alias(f"sell_price_lagged_{lag}")
                for lag in PRICE_LAGS
            ]
            + [
                pl.col("sell_price").shift(1)
                .rolling_mean(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sell_price_rolling_mean_{window}")
                .cast(pl.Float32)
                for window in ROLLING_PRICE_WINDOWS
            ]
            + [
                pl.col("sell_price").shift(1)
                .rolling_std(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sell_price_rolling_std_{window}")
                .cast(pl.Float32)
                for window in ROLLING_PRICE_WINDOWS
            ]
        )
        .drop("sell_price")
        .filter(pl.col("d_index") == d_index)
    )
    # Keep the price history bounded
    price_history_df = price_history_df.filter(pl.col("d_index") > d_index - max_lookback)

    # Join id, sales, price and calendar features
    test_df = (
        test_df.drop("sales")
        .join(test_sales_features_df, on=["id", "d_index"], how="inner", validate="1:1")
        .join(test_price_features_df, on=["id", "d_index"], how="inner", validate="1:1")
        .join(forecast_calendar_df, on="d_index", how="inner", validate="m:1")
        # Series without a price at this timestamp aren't sold: no forecast
        .filter(pl.col("sell_price").is_not_null())
    )

    # Make predictions. The poisson objective makes `predict` return the expected count
    # (the inverse link is applied). Use the best iteration found by early stopping.
    y_hat = model.predict(to_float32_matrix(test_df), num_iteration=model.best_iteration)

    # Predictions are aligned with the rows of test_df. Sales can't be negative, and are
    # rounded to integers so they can be fed back in as lagged sales features.
    y_hat_df = (
        pl.DataFrame({"id": test_df["id"].cast(pl.String), "sales": y_hat})
        .with_columns(
            sales=pl.col("sales").clip(lower_bound=0).round().cast(pl.Int16),
            d_index=pl.lit(d_index).cast(pl.Int16),
            F=pl.format("F{}", f_index),
        )
    )
    all_forecast_dfs.append(y_hat_df)

In [ ]:
final_forecast_df = (
    pl.concat(all_forecast_dfs)
    .with_columns(sales=pl.col("sales").cast(pl.Int64))
    .pivot(on="F", index="id", values="sales")
)

submission_df = (
    sample_submission_df
    .join(final_forecast_df, on="id", how="left", suffix="_model", validate="1:1")
    .with_columns(
        pl.coalesce(pl.col(f"{c}_model"), pl.col(c)).cast(pl.Int64).alias(c)
        for c in SUBMISSION_F_COLS
    )
    .select(sample_submission_df.columns)
)

In [ ]:
submission_df.tail()